In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder,OrdinalEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

2026-06-23 06:37:37.199194: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-23 06:37:37.207780: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-23 06:37:37.324777: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-23 06:37:37.324897: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-23 06:37:37.328600: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

In [2]:
df = pd.read_csv('Churn_Modelling.csv')
df

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [3]:
## Preprocess the data
### Drop irrelevent columns

df = df.drop(['RowNumber','CustomerId','Surname'],axis = 1)

# Dividing the dataset into train & test

X = df.drop('Exited',axis=1)
y = df['Exited']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state=42)



# Encoding categorical data

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

# ColumnTranform expects column name not the dataset!!
preprocessor = ColumnTransformer( 
    transformers=[
        ('binary', OrdinalEncoder(categories=[['Female','Male']]), ['Gender']),
        ('ohe',OneHotEncoder(sparse_output=False),['Geography']),
        ('Scale',StandardScaler(),['CreditScore','Age','Tenure','Balance','NumOfProducts','HasCrCard','IsActiveMember','EstimatedSalary'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)


# Transforming the columns (array!!)
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)




In [4]:
# Define a function to create the model and try different Parameters(KerasClassifier)

def create_model(neurons = 32,layers=1):
    model = Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train_transformed.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss = "binary_crossentropy",metrics = ['accuracy'])

    return model

In [5]:
## Create KerasClassifier

model = KerasClassifier(layers=1,neurons=32,build_fn=create_model,epochs=50,verbose=1)

In [6]:
## Define the grid search parameters

param_grid = {
    'neurons' : [16,32,64,128],
    'layers' : [1,2],
    'epochs' : [50,100]
}

In [7]:
## Perform grid Search

grid = GridSearchCV(estimator=model,param_grid=param_grid,n_jobs=-1,cv=3,verbose=1)
grid_result = grid.fit(X_train_transformed,y_train)

# Print the best parameters
print("Best: %f using %s" % (grid_result.best_score_,grid_result.best_params_ ))

Fitting 3 folds for each of 16 candidates, totalling 48 fits


2026-06-23 06:37:48.986561: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-23 06:37:48.996717: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-23 06:37:49.232988: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-23 06:37:49.234409: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-23 06:37:49.240397: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

Epoch 1/50


/home/ronit/1PROJECTS/ann_project/ann_env/lib/python3.11/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/home/ronit/1PROJECTS/ann_project/ann_env/lib/python3.11/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/home/ronit/1PROJECTS/ann_project/ann_env/lib/python3.11/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/home/ronit/1PROJECTS/ann_project/ann_env/lib/python3.11/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use o

Epoch 1/50
Epoch 1/50


...

160/167 [===========================>..] - ETA: 0s - loss: 0.2885 - accuracy: 0.8748Epoch 1/100
Epoch 25/100
167/167 [==============================] - 2s 12ms/step - loss: 0.3236 - accuracy: 0.8665
Epoch 27/100
167/167 [==============================] - 2s 11ms/step - loss: 0.2789 - accuracy: 0.8825
Epoch 98/100
167/167 [==============================] - 2s 11ms/step - loss: 0.3142 - accuracy: 0.8661
Epoch 26/100
167/167 [==============================] - 2s 12ms/step - loss: 0.3181 - accuracy: 0.8674
Epoch 26/100
167/167 [==============================] - 2s 13ms/step - loss: 0.3025 - accuracy: 0.8744
Epoch 24/100
167/167 [==============================] - 2s 12ms/step - loss: 0.2776 - accuracy: 0.8843
Epoch 99/100
167/167 [==============================] - 2s 12ms/step - loss: 0.3140 - accuracy: 0.8672
Epoch 27/100
167/167 [==============================] - 2s 12ms/step - loss: 0.3232 - accuracy: 0.8657
Epoch 29/100
167/167 [==============================] - 2s 11ms/step - loss: 0.3

/home/ronit/1PROJECTS/ann_project/ann_env/lib/python3.11/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


167/167 [==============================] - 2s 12ms/step - loss: 0.2941 - accuracy: 0.8757
Epoch 28/100
167/167 [==============================] - 2s 11ms/step - loss: 0.2908 - accuracy: 0.8802
Epoch 22/100
Epoch 24/100
 21/167 [==>...........................] - ETA: 2s - loss: 0.2703 - accuracy: 0.8824Epoch 27/100
Epoch 31/100
167/167 [==============================] - 2s 12ms/step - loss: 0.3167 - accuracy: 0.8716
Epoch 30/100
167/167 [==============================] - 2s 12ms/step - loss: 0.3084 - accuracy: 0.8729
Epoch 30/100
167/167 [==============================] - 2s 14ms/step - loss: 0.2736 - accuracy: 0.8815
Epoch 23/100
167/167 [==============================] - 3s 16ms/step - loss: 0.2942 - accuracy: 0.8783
Epoch 28/100
167/167 [==============================] - 2s 13ms/step - loss: 0.3098 - accuracy: 0.8750
Epoch 31/100
167/167 [==============================] - 2s 13ms/step - loss: 0.2690 - accuracy: 0.8832
Epoch 6/100
Epoch 33/100
167/167 [==============================] 

/home/ronit/1PROJECTS/ann_project/ann_env/lib/python3.11/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


Epoch 1/50
250/250 [==============================] - 2s 3ms/step - loss: 0.4645 - accuracy: 0.7945
Epoch 2/50
250/250 [==============================] - 1s 3ms/step - loss: 0.3993 - accuracy: 0.8326
Epoch 3/50
250/250 [==============================] - 1s 3ms/step - loss: 0.3734 - accuracy: 0.8494
Epoch 4/50
250/250 [==============================] - 1s 3ms/step - loss: 0.3593 - accuracy: 0.8546
Epoch 5/50
250/250 [==============================] - 1s 3ms/step - loss: 0.3511 - accuracy: 0.8570
Epoch 6/50
250/250 [==============================] - 1s 3ms/step - loss: 0.3466 - accuracy: 0.8566
Epoch 7/50
250/250 [==============================] - 1s 3ms/step - loss: 0.3430 - accuracy: 0.8576
Epoch 8/50
250/250 [==============================] - 1s 3ms/step - loss: 0.3414 - accuracy: 0.8555
Epoch 9/50
250/250 [==============================] - 1s 3ms/step - loss: 0.3398 - accuracy: 0.8581
Epoch 10/50
250/250 [==============================] - 1s 3ms/step - loss: 0.3377 - accuracy: 0.8591